# 实验 02 · 理解样本、奖励与词元掩码

本课直接导入固定版本 slime.utils.types.Sample，而非自定义一个同名替身。只使用 CPU；token ID 和 log-prob 为明确标注的人工测试夹具，不是模型生成结果。

单样本奖励接口为 `async def custom_rm(args, sample) -> float`；使用 `--group-rm` 时，应提供接受样本列表、返回等长分数列表的函数。我们的两个入口分别是 `course.strict_reward` 和 `course.batch_reward`。只有课程目录位于所有 worker 的 PYTHONPATH 时，路径才能在远端解析。

## 1 · 奖励是协议，不是字符串包含测试

只接受去除首尾空白、转小写后的 yes/no，必须与 label 完全匹配。错误标签视为数据错误并抛出异常，不能悄悄记零分。此协议不要求 EOS；停止行为需要另外检查。


In [ ]:
from types import SimpleNamespace
import course
Sample=course.load_sample()
args=SimpleNamespace()
cases=[('yes','yes',1),('YES ','yes',1),('no','yes',0),('yes no','yes',0),('yes because...','yes',0),('','yes',0)]
samples=[Sample(response=answer,label=label) for answer,label,_ in cases]
scores=await course.batch_reward(args,samples)
assert scores==[expected for _,_,expected in cases]
for case,score in zip(cases,scores):print(repr(case[0]),score)
try:
    await course.strict_reward(args,Sample(response='yes',label='invalid'))
except ValueError:print('PASS: 错误标签被显式拒绝。')
else:raise AssertionError('Invalid dataset label accepted')


## 2 · response_length 与 loss_mask

tokens 包含提示及响应 token。response_length 指响应部分长度，不是字符数。loss_mask 与响应位置对齐，不能误用全序列长度。工具结果可以位于响应轨迹内，但它不是策略采样的动作，不应直接作为策略损失 token。

下格构造两个 prompt token 和四个响应位置；中间两个位置模拟工具返回，mask 为零。rollout_log_probs 的工具位置使用占位值，只在 mask 下排除。本格只检查课程夹具的对齐，复杂多轮生成需遵循实际接口并保留生成 token 的概率与权重版本。


In [ ]:
sample=Sample(group_index=0,index=0,rollout_id=0,prompt='Synthetic prompt',
    tokens=[10,11,20,30,31,21],response='Synthetic trajectory',response_length=4,
    label='yes',loss_mask=[1,0,0,1],rollout_log_probs=[-.2,0.,0.,-.3],
    status=Sample.Status.COMPLETED)
alignment=course.audit(sample)
assert alignment=={'prompt_tokens':2,'response_tokens':4,'loss_tokens':2}
print(alignment)
sample.loss_mask=[1,1]
try:course.audit(sample)
except AssertionError:print('PASS: 错位 mask 被拒绝。')
else:raise AssertionError('Misaligned mask accepted')
sample.loss_mask=[1,0,0,1]


## 3 · 同一道题、多个 rollout、多个 segment

group_index 表示用于相对比较的提示组。一个完整 rollout 可能被拆成多个训练 segment；固定版本提供 rollout_id 来标识同一次完整执行。如果每个 segment 都使用新 rollout_id，就可能改变组统计与损失权重。

官方定制文档说明：同一完整轨迹只有一个结果奖励时，各 segment 使用相同奖励并保留 group_index 与 rollout_id；不要自行按 segment 数再除一次奖励。共享生成前缀也不能重复计入损失。实现细节必须与所固定版本一致。

下格仅演示去重后的完整轨迹奖励，不调用 slime 的优势实现。它展示为什么把 segment 当成独立回答会改变平均值。


In [ ]:
segments=[dict(group=0,rollout=100,reward=1.),dict(group=0,rollout=100,reward=1.),dict(group=0,rollout=101,reward=0.)]
unique={}
for s in segments:
    key=(s['group'],s['rollout'])
    if key in unique:assert unique[key]==s['reward']
    unique[key]=s['reward']
wrong=sum(s['reward'] for s in segments)/len(segments)
right=sum(unique.values())/len(unique)
assert right==.5 and wrong!=right
print('按 segment 误算:',wrong,'按完整轨迹:',right)
course.save_report('02_contracts.json',dict(real_upstream_sample=True,reward_cases=len(cases),
    invalid_label_rejected=True,misaligned_mask_rejected=True,alignment=alignment,
    segment_mean=wrong,rollout_mean=right,scope='CPU interface tests with synthetic tokens; no actual rollout or training.'))


## 4 · 课后验收

为截断、空响应、只有工具返回、奖励超时分别定义处理规则。Sample 的 COMPLETED、TRUNCATED、ABORTED、FAILED 不应被一律归为“模型答错”；系统失败率与任务准确率要分开报告。

再尝试把奖励函数改成“包含 yes 就得分”，预测哪条攻击样例会通过。不要把这项修改带入正式评分。下一课用这些契约设计完整训练的调试顺序。


---

<a href="https://github.com/SergioTermann"><img src="assets/logo.png" alt="SergioTermann" width="44" height="44" align="left" style="margin: 0 12px 0 0;"></a>

**更多有趣的 AI 课程**  
[← 访问 SergioTermann 的 GitHub 主页](https://github.com/SergioTermann)